# System klasyfikacji zgłoszeń i wiadomości użytkowników

## Cel projektu

Celem projektu jest zbudowanie prototypu systemu, który automatycznie przypisuje wiadomości/zgłoszenia użytkowników (np. studentów) do jednej z predefiniowanych kategorii:

- **stypendium** – zapytania o stypendium socjalne, naukowe, rektora itp.
- **rekrutacja** – zapytania dotyczące przyjęcia na studia, dokumentów, terminów rekrutacji
- **plan_zajec** – pytania o harmonogram, zmiany grup, sale, godziny
- **awaria_systemu** – zgłoszenia problemów technicznych (USOS, e-mail, VPN, platforma)
- **sprawy_finansowe** – opłaty za studia, faktury, przelewy, zwroty
- **inne** – pozostałe zapytania

## Model

Używany model to **`speakleash/Bielik-1.5B-v3.0-Instruct`** – polski model językowy typu decoder, dostępny na Hugging Face:  
https://huggingface.co/speakleash/Bielik-1.5B-v3.0-Instruct

## Dane

1. **Dane własne** – 180 przykładowych zgłoszeń studenckich w języku polskim (30 na kategorię), podzielonych na zbiór few-shot (5 na kategorię) i testowy (25 na kategorię).
2. **`allegro/klej-polemo2-in`** – publiczny polski zbiór recenzji z etykietami sentymentu, używany jako **zewnętrzny punkt odniesienia**: sprawdzamy, czy model nie myli tekstów spoza domeny uczelni z kategoriami helpdesku.

## Techniki

Projekt realizuje dwa podejścia i porównuje ich skuteczność:

1. **Few-shot prompting** – w prompcie podajemy przykłady każdej kategorii, model klasyfikuje bez trenowania
2. **QLoRA fine-tuning** – dotrenowujemy model na własnych danych przy użyciu metody LoRA (Low-Rank Adaptation) w 4-bitowej kwantyzacji.

Model jest ładowany w **4-bitowej kwantyzacji (NF4)** przy użyciu `BitsAndBytesConfig`, co zmniejsza zużycie pamięci GPU ~4× względem FP16.

## 1. Instalacja pakietów oraz konfiguracja tokenu do połączenia z Hugging Face

In [1]:
# Konfiguracja użycia tokenu do Hugging Face używając Colab Secrets
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get("HF_TOKEN")
if hf_token:
  login(token=hf_token)
  print("Poprawny token, zalogowano się do Hugging Face")
else:
  print("Brak tokenu HF_TOKEN w Colab Secrets")

Poprawny token, zalogowano się do Hugging Face


In [2]:
# Instalacja potrzebnych pakietów
!pip install -q -U transformers datasets accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 73.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 16.8 MB/s eta 0:00:00


## 2. Importy i sprawdzenie GPU

In [3]:
# Import pakietów oraz sprawdzenie czy używamy GPU
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sklearn.metrics import classification_report

print("PyTorch:", torch.__version__)
print("CUDA dostępne:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

device = "cuda" if torch.cuda.is_available() else "cpu"

PyTorch: 2.11.0+cu128
CUDA dostępne: True
GPU: Tesla T4


## 3. Zbiór danych

Własnoręcznie przygotowany zbiór180 przykładowych zgłoszeń studenckich w języku polskim.
Każda kategoria zawiera 30 przykładów.

In [4]:
# Własny zbiór danych, łącznie 180 pytań po 30 na kategorie

data = [
    # ------------------------------------------------------------------ stypendium (30)
    {"text": "Chciałbym zapytać o wymagania dotyczące stypendium rektora na nowy rok akademicki.", "label": "stypendium"},
    {"text": "Kiedy można składać wnioski o stypendium socjalne na semestr letni?", "label": "stypendium"},
    {"text": "Nie otrzymałem decyzji o stypendium naukowym mimo złożonego wniosku.", "label": "stypendium"},
    {"text": "Proszę o informację, jak odwołać się od decyzji komisji stypendialnej.", "label": "stypendium"},
    {"text": "Czy stypendium dla najlepszych studentów obejmuje też studentów pierwszego roku?", "label": "stypendium"},
    {"text": "Jaki dochód na osobę uprawnia do stypendium socjalnego?", "label": "stypendium"},
    {"text": "Czy mogę złożyć wniosek o stypendium specjalne dla niepełnosprawnych?", "label": "stypendium"},
    {"text": "Moje stypendium nie wpłynęło na konto w wyznaczonym terminie.", "label": "stypendium"},
    {"text": "Proszę o informację o stypendiach zewnętrznych dla doktorantów.", "label": "stypendium"},
    {"text": "Ile wynosi maksymalna kwota stypendium rektora w bieżącym roku?", "label": "stypendium"},
    {"text": "Jakie dokumenty należy dołączyć do wniosku o stypendium socjalne?", "label": "stypendium"},
    {"text": "Czy mogę pobierać stypendium socjalne i stypendium rektora jednocześnie?", "label": "stypendium"},
    {"text": "Kiedy komisja stypendialna ogłosi wyniki rozpatrywania wniosków?", "label": "stypendium"},
    {"text": "Czy zmiana kierunku studiów wpływa na utratę przyznanego stypendium?", "label": "stypendium"},
    {"text": "Proszę o wzór wniosku o stypendium dla osób w trudnej sytuacji materialnej.", "label": "stypendium"},
    {"text": "Czy stypendium jest wypłacane za sierpień podczas wakacji letnich?", "label": "stypendium"},
    {"text": "Gdzie składam zaświadczenie o dochodach do wniosku stypendialnego?", "label": "stypendium"},
    {"text": "Czy urlop dziekański przerywa wypłatę stypendium socjalnego?", "label": "stypendium"},
    {"text": "Jak obliczyć dochód na osobę w rodzinie do celów stypendialnych?", "label": "stypendium"},
    {"text": "Dostałem odmowę stypendium – czy przysługuje mi ponowne rozpatrzenie?", "label": "stypendium"},
    {"text": "Czy stypendium rektora przysługuje studentom studiów niestacjonarnych?", "label": "stypendium"},
    {"text": "Jak długo trwa rozpatrywanie wniosku o stypendium socjalne?", "label": "stypendium"},
    {"text": "Czy mogę składać wniosek stypendialny przez internet, czy tylko osobiście?", "label": "stypendium"},
    {"text": "Stypendium zostało mi cofnięte po wznowieniu studiów – dlaczego?", "label": "stypendium"},
    {"text": "Proszę o informację, czy stypendium sportowe można łączyć ze stypendium rektora.", "label": "stypendium"},
    {"text": "Czy student powtarzający rok może ubiegać się o stypendium socjalne?", "label": "stypendium"},
    {"text": "Kiedy upływa termin złożenia wniosku o stypendium na semestr zimowy?", "label": "stypendium"},
    {"text": "Proszę o zaświadczenie potwierdzające przyznanie stypendium rektora.", "label": "stypendium"},
    {"text": "Czy student drugiego stopnia może otrzymać stypendium naukowe?", "label": "stypendium"},
    {"text": "Na jakie konto jest przelewane stypendium i czy można je zmienić?", "label": "stypendium"},

    # ------------------------------------------------------------------ rekrutacja (30)
    {"text": "Jakie dokumenty należy złożyć podczas rekrutacji na studia magisterskie?", "label": "rekrutacja"},
    {"text": "Kiedy ruszają zapisy na studia podyplomowe z data science?", "label": "rekrutacja"},
    {"text": "Czy wynik matury z matematyki jest brany pod uwagę przy rekrutacji na informatykę?", "label": "rekrutacja"},
    {"text": "Nie mogę zalogować się do systemu IRK podczas rejestracji.", "label": "rekrutacja"},
    {"text": "Jakie są progi punktowe na kierunek elektronika w zeszłym roku?", "label": "rekrutacja"},
    {"text": "Czy cudzoziemcy mogą ubiegać się o przyjęcie na studia w języku polskim?", "label": "rekrutacja"},
    {"text": "Do kiedy można uzupełnić dokumenty po dostaniu się na listę rezerwową?", "label": "rekrutacja"},
    {"text": "Chcę przenieść się z innej uczelni – jak wygląda procedura?", "label": "rekrutacja"},
    {"text": "Czy wyniki olimpiad technicznych zwiększają szanse przy rekrutacji?", "label": "rekrutacja"},
    {"text": "Kiedy zostaną ogłoszone wyniki pierwszej tury rekrutacji?", "label": "rekrutacja"},
    {"text": "Czy muszę stawić się osobiście w dziekanacie, aby potwierdzić przyjęcie?", "label": "rekrutacja"},
    {"text": "Ile miejsc jest dostępnych na kierunku automatyka i robotyka?", "label": "rekrutacja"},
    {"text": "Czy można aplikować na dwa kierunki jednocześnie w tej samej rekrutacji?", "label": "rekrutacja"},
    {"text": "Kiedy startuje rekrutacja uzupełniająca na wolne miejsca?", "label": "rekrutacja"},
    {"text": "Jakie są wymagania językowe dla kandydatów na studia anglojęzyczne?", "label": "rekrutacja"},
    {"text": "Czy przy rekturacji dyplom z innego kraju musi być przetłumaczony przez tłumacza przysięgłego?", "label": "rekrutacja"},
    {"text": "Proszę o informację o wymaganym poziomie języka angielskiego na kierunek MBA.", "label": "rekrutacja"},
    {"text": "Jak przebiega egzamin wstępny na kierunek architektura?", "label": "rekrutacja"},
    {"text": "Czy opłata rekrutacyjna jest zwracana w przypadku nieprzyjęcia na studia?", "label": "rekrutacja"},
    {"text": "Kiedy muszę złożyć oryginał świadectwa maturalnego?", "label": "rekrutacja"},
    {"text": "Jak sprawdzić swoje miejsce na liście rankingowej kandydatów?", "label": "rekrutacja"},
    {"text": "Czy mogę zmienić wybrany kierunek po opłaceniu wpisowego?", "label": "rekrutacja"},
    {"text": "Jakie przedmioty są objęte egzaminem wstępnym na kierunek lekarski?", "label": "rekrutacja"},
    {"text": "Czy wymagana jest rozmowa kwalifikacyjna przy rekrutacji na studia doktoranckie?", "label": "rekrutacja"},
    {"text": "Jak wygląda procedura przyjęcia na studia dla laureatów olimpiad przedmiotowych?", "label": "rekrutacja"},
    {"text": "Czy mogę zapisać się na listę rezerwową po ogłoszeniu wyników?", "label": "rekrutacja"},
    {"text": "Kiedy uczelnia poinformuje mnie o wynikach rozmowy kwalifikacyjnej?", "label": "rekrutacja"},
    {"text": "Jakie dokumenty potrzeba do rekrutacji jako kandydat z orzeczeniem o niepełnosprawności?", "label": "rekrutacja"},
    {"text": "Czy aplikacja złożona po terminie zostanie rozpatrzona?", "label": "rekrutacja"},
    {"text": "Gdzie mogę sprawdzić, czy moje dokumenty rekrutacyjne zostały przyjęte?", "label": "rekrutacja"},

    # ------------------------------------------------------------------ plan_zajec (30)
    {"text": "Kiedy będzie dostępny plan zajęć na semestr letni?", "label": "plan_zajec"},
    {"text": "W USOS widzę konflikt godzinowy między dwoma przedmiotami obowiązkowymi.", "label": "plan_zajec"},
    {"text": "Czy jest możliwa zmiana grupy laboratoryjnej z poniedziałkowej na środową?", "label": "plan_zajec"},
    {"text": "Proszę o informację, w której sali odbywają się ćwiczenia z analizy matematycznej.", "label": "plan_zajec"},
    {"text": "Wykład z baz danych został przeniesiony – kiedy i gdzie?", "label": "plan_zajec"},
    {"text": "Nie mogę znaleźć w systemie planu zajęć dla mojego roku.", "label": "plan_zajec"},
    {"text": "Jakie są godziny dyżurów asystentów z fizyki?", "label": "plan_zajec"},
    {"text": "Czy w tygodniu rektorskim odbywają się normalne zajęcia?", "label": "plan_zajec"},
    {"text": "Proszę o wyjaśnienie, dlaczego mam dwa wykłady w tej samej godzinie.", "label": "plan_zajec"},
    {"text": "Kiedy odbędzie się egzamin poprawkowy z programowania?", "label": "plan_zajec"},
    {"text": "Czy zajęcia z wychowania fizycznego są obowiązkowe na pierwszym roku?", "label": "plan_zajec"},
    {"text": "Jak zapisać się na lektorat z języka niemieckiego w nowym semestrze?", "label": "plan_zajec"},
    {"text": "Czy mogę uczestniczyć w wykładzie innej grupy, jeśli mam kolizję w planie?", "label": "plan_zajec"},
    {"text": "W planie nie ma żadnych zajęć w piątek – czy to błąd systemu?", "label": "plan_zajec"},
    {"text": "Proszę o aktualny harmonogram sesji egzaminacyjnej zimowej.", "label": "plan_zajec"},
    {"text": "Kiedy zostanie opublikowany terminarz zaliczeń na koniec semestru?", "label": "plan_zajec"},
    {"text": "Czy prowadzący może odwołać zajęcia bez podania powodu?", "label": "plan_zajec"},
    {"text": "Jak zgłosić nieobecność na zajęciach obowiązkowych z powodu choroby?", "label": "plan_zajec"},
    {"text": "Czy istnieje możliwość zapisu na seminarium dyplomowe w trakcie semestru?", "label": "plan_zajec"},
    {"text": "Proszę o informację o terminie obrony prac dyplomowych w czerwcu.", "label": "plan_zajec"},
    {"text": "Ile razy mogę przystąpić do egzaminu z danego przedmiotu?", "label": "plan_zajec"},
    {"text": "Czy zajęcia terenowe wliczają się do puli obowiązkowych godzin?", "label": "plan_zajec"},
    {"text": "Kiedy jest ostateczny termin wpisania ocen przez prowadzącego do USOS?", "label": "plan_zajec"},
    {"text": "Czy mogę złożyć wniosek o zmianę terminu egzaminu ze względu na kolizję?", "label": "plan_zajec"},
    {"text": "Proszę o podanie sali, w której odbywają się ćwiczenia z chemii organicznej.", "label": "plan_zajec"},
    {"text": "Kiedy są przewidziane odrabianie zajęć odwołanych w tygodniu rektorskim?", "label": "plan_zajec"},
    {"text": "Czy plan zajęć zostanie zmieniony po powrocie wykładowcy z urlopu?", "label": "plan_zajec"},
    {"text": "Nie widzę żadnych zajęć przypisanych do mojego konta w USOSweb.", "label": "plan_zajec"},
    {"text": "Proszę o informację, czy egzamin dyplomowy odbywa się w sesji czy poza nią.", "label": "plan_zajec"},
    {"text": "Kiedy ogłoszony zostanie plan zajęć dla nowo przyjętych studentów?", "label": "plan_zajec"},

    # ------------------------------------------------------------------ awaria_systemu (30)
    {"text": "Nie mogę zalogować się do USOS od rana, pojawia się błąd 500.", "label": "awaria_systemu"},
    {"text": "Platforma e-learningowa Moodle nie wczytuje materiałów do kursu.", "label": "awaria_systemu"},
    {"text": "VPN uczelniany nie działa na moim komputerze po aktualizacji systemu.", "label": "awaria_systemu"},
    {"text": "Otrzymałem błąd przy próbie zapisania się na egzamin w USOSweb.", "label": "awaria_systemu"},
    {"text": "Moje konto pocztowe na domenie uczelni zostało zablokowane.", "label": "awaria_systemu"},
    {"text": "System rezerwacji sal nie pozwala mi zarezerwować sali na projekt grupowy.", "label": "awaria_systemu"},
    {"text": "Nie dostaję powiadomień e-mail z systemu dziekanatowego.", "label": "awaria_systemu"},
    {"text": "Certyfikat SSL strony uczelnianej wygasł i przeglądarka blokuje dostęp.", "label": "awaria_systemu"},
    {"text": "Aplikacja mobilna uczelni crasha przy próbie sprawdzenia ocen.", "label": "awaria_systemu"},
    {"text": "Baza biblioteczna nie odpowiada, nie mogę sprawdzić dostępności książki.", "label": "awaria_systemu"},
    {"text": "System do składania wniosków działa bardzo wolno i przekracza limit czasu.", "label": "awaria_systemu"},
    {"text": "Nie mogę pobrać zaświadczenia o studiowaniu z USOSweb – strona się nie ładuje.", "label": "awaria_systemu"},
    {"text": "Po zmianie hasła nie mogę zalogować się do żadnego systemu uczelnianego.", "label": "awaria_systemu"},
    {"text": "Wideokonferencja na Microsoft Teams uczelnianym nie uruchamia się.", "label": "awaria_systemu"},
    {"text": "Nie widzę moich ocen w USOS mimo zakończonej sesji egzaminacyjnej.", "label": "awaria_systemu"},
    {"text": "Konto w systemie antyplagiatowym zostało dezaktywowane przed obroną.", "label": "awaria_systemu"},
    {"text": "Serwer poczty uczelnianej odrzuca wychodzące wiadomości e-mail.", "label": "awaria_systemu"},
    {"text": "Portal studenta zwraca błąd 404 dla strony mojego profilu.", "label": "awaria_systemu"},
    {"text": "Nie mogę przesłać pliku z pracą dyplomową przez platformę APD – limit rozmiaru?", "label": "awaria_systemu"},
    {"text": "System zapisów na lektoraty zawiesza się po kliknięciu 'Zapisz'.", "label": "awaria_systemu"},
    {"text": "Drukarka w czytelni bibliotecznej pokazuje błąd połączenia z serwerem.", "label": "awaria_systemu"},
    {"text": "Dostęp do dziennika elektronicznego jest zablokowany od wczoraj.", "label": "awaria_systemu"},
    {"text": "Logowanie przez SSO przestało działać po weekendzie.", "label": "awaria_systemu"},
    {"text": "System płatności uczelnianej zwraca błąd przy próbie opłacenia czesnego online.", "label": "awaria_systemu"},
    {"text": "Nie widzę przypisanych do mnie przedmiotów w systemie Moodle.", "label": "awaria_systemu"},
    {"text": "Wi-Fi eduroam nie działa na terenie budynku głównego wydziału.", "label": "awaria_systemu"},
    {"text": "Zniknął mi dostęp do repozytorium GitLab uczelnianego.", "label": "awaria_systemu"},
    {"text": "Po reinstalacji systemu Windows nie mogę zainstalować oprogramowania licencjonowanego przez uczelnię.", "label": "awaria_systemu"},
    {"text": "Moja praca domowa przesłana przez Moodle nie dotarła do prowadzącego.", "label": "awaria_systemu"},
    {"text": "Kamera i mikrofon nie działają podczas egzaminu zdalnego przez Teams.", "label": "awaria_systemu"},

    # ------------------------------------------------------------------ sprawy_finansowe (30)
    {"text": "Kiedy należy uiścić opłatę za drugi semestr studiów niestacjonarnych?", "label": "sprawy_finansowe"},
    {"text": "Proszę o wystawienie faktury VAT za czesne za rok akademicki.", "label": "sprawy_finansowe"},
    {"text": "Czy mogę uzyskać rozłożenie opłaty za studia na raty?", "label": "sprawy_finansowe"},
    {"text": "Na konto uczelni wpłaciłem za dużo – jak uzyskać zwrot nadpłaty?", "label": "sprawy_finansowe"},
    {"text": "Nie pojawia się należność do zapłaty w systemie mimo mojego statusu studiów płatnych.", "label": "sprawy_finansowe"},
    {"text": "Jaki jest numer konta do wpłaty opłaty rekrutacyjnej?", "label": "sprawy_finansowe"},
    {"text": "Proszę o potwierdzenie zaksięgowania wpłaty czesnego za październik.", "label": "sprawy_finansowe"},
    {"text": "Czy opłata za powtarzanie semestru obowiązuje od razu czy po decyzji dziekana?", "label": "sprawy_finansowe"},
    {"text": "Chciałbym uzyskać zaświadczenie o niezaleganiu z opłatami.", "label": "sprawy_finansowe"},
    {"text": "Ile wynosi opłata za wydanie duplikatu legitymacji studenckiej?", "label": "sprawy_finansowe"},
    {"text": "Czy mogę zapłacić czesne kartą płatniczą w kasie uczelni?", "label": "sprawy_finansowe"},
    {"text": "Kiedy zostanie wystawiona faktura za czesne za semestr letni?", "label": "sprawy_finansowe"},
    {"text": "Czy przerwa w studiach (urlop) zwalnia z obowiązku opłaty czesnego?", "label": "sprawy_finansowe"},
    {"text": "Jakie są konsekwencje finansowe niezaliczenia semestru na studiach płatnych?", "label": "sprawy_finansowe"},
    {"text": "Proszę o informację o opłacie za wydanie suplementu do dyplomu w języku angielskim.", "label": "sprawy_finansowe"},
    {"text": "Czy jest możliwe umorzenie zaległości w płatności czesnego ze względu na trudną sytuację?", "label": "sprawy_finansowe"},
    {"text": "Ile kosztuje egzamin komisyjny i jak go opłacić?", "label": "sprawy_finansowe"},
    {"text": "Proszę o wyciąg z konta rozliczeniowego moich wpłat na uczelnię.", "label": "sprawy_finansowe"},
    {"text": "Kiedy upływa termin płatności czesnego bez naliczania odsetek?", "label": "sprawy_finansowe"},
    {"text": "Czy doktorant pobierający stypendium musi płacić czesne?", "label": "sprawy_finansowe"},
    {"text": "Jak sprawdzić aktualne saldo mojego konta studenckiego w systemie?", "label": "sprawy_finansowe"},
    {"text": "Jaka jest opłata za przepisanie oceny z poprzedniego kierunku studiów?", "label": "sprawy_finansowe"},
    {"text": "Czy mogę zapłacić czesne w ratach miesięcznych zamiast semestralnych?", "label": "sprawy_finansowe"},
    {"text": "Proszę o korektę faktury – błędnie wpisano moje nazwisko.", "label": "sprawy_finansowe"},
    {"text": "Ile wynosi opłata za wznowienie studiów po skreśleniu z listy studentów?", "label": "sprawy_finansowe"},
    {"text": "Czy opłata za obronę pracy dyplomowej jest wliczona w czesne?", "label": "sprawy_finansowe"},
    {"text": "Skąd mam pobrać indywidualny numer konta do wpłat czesnego?", "label": "sprawy_finansowe"},
    {"text": "Czy uczelnia wystawia PIT-11 za wypłacone stypendium?", "label": "sprawy_finansowe"},
    {"text": "Jak uzyskać potwierdzenie płatności czesnego do urzędu skarbowego?", "label": "sprawy_finansowe"},
    {"text": "Proszę o informację, czy opłata wpisowa przy rekrutacji jest zaliczana na poczet czesnego.", "label": "sprawy_finansowe"},

    # ------------------------------------------------------------------ inne (30)
    {"text": "Kiedy jest następny dzień otwarty uczelni dla maturzystów?", "label": "inne"},
    {"text": "Proszę o kontakt do opiekuna roku na kierunku mechatronika.", "label": "inne"},
    {"text": "Gdzie mogę znaleźć regulamin studiów?", "label": "inne"},
    {"text": "Czy na kampusie jest dostępna stołówka w godzinach wieczornych?", "label": "inne"},
    {"text": "Jak uzyskać kartę dostępu do laboratoriów wydziałowych?", "label": "inne"},
    {"text": "Proszę o informację o kołach naukowych działających na wydziale.", "label": "inne"},
    {"text": "Czy mogę wydrukować pracę dyplomową w bibliotece uczelnianej?", "label": "inne"},
    {"text": "Kiedy są godziny otwarcia dziekanatu?", "label": "inne"},
    {"text": "Proszę o informację o możliwości odbycia praktyk zawodowych za granicą.", "label": "inne"},
    {"text": "Jak złożyć podanie o urlop dziekański?", "label": "inne"},
    {"text": "Gdzie można znaleźć aktualny regulamin biblioteki uczelnianej?", "label": "inne"},
    {"text": "Jak uzyskać zaświadczenie o byciu studentem do ZUS?", "label": "inne"},
    {"text": "Czy istnieje możliwość zakwaterowania w akademiku dla studentów pierwszego roku?", "label": "inne"},
    {"text": "Jak zapisać się do uczelnianej przychodni zdrowia?", "label": "inne"},
    {"text": "Proszę o informację, czy uczelnia oferuje zniżki na transport publiczny.", "label": "inne"},
    {"text": "Jak uzyskać duplikat karty bibliotecznej?", "label": "inne"},
    {"text": "Gdzie zgłaszam zagubioną legitymację studencką?", "label": "inne"},
    {"text": "Czy studenci mają dostęp do siłowni uczelnianej bezpłatnie?", "label": "inne"},
    {"text": "Jak mogę zarezerwować pokój w akademiku na rok akademicki?", "label": "inne"},
    {"text": "Proszę o informację o programach wymiany studenckiej Erasmus+.", "label": "inne"},
    {"text": "Gdzie mogę uzyskać pomoc psychologiczną oferowaną przez uczelnię?", "label": "inne"},
    {"text": "Jak wygląda procedura rezygnacji ze studiów?", "label": "inne"},
    {"text": "Czy biblioteka uczelniana wypożycza laptopy studentom?", "label": "inne"},
    {"text": "Proszę o informację o wolontariacie organizowanym przez samorząd studencki.", "label": "inne"},
    {"text": "Kiedy odbędzie się inauguracja roku akademickiego?", "label": "inne"},
    {"text": "Jak złożyć skargę na prowadzącego zajęcia?", "label": "inne"},
    {"text": "Czy uczelnia organizuje kursy języka polskiego dla cudzoziemców?", "label": "inne"},
    {"text": "Proszę o informację o miejscach parkingowych dla studentów na terenie uczelni.", "label": "inne"},
    {"text": "Jak uzyskać dostęp do zasobów cyfrowych biblioteki z domu?", "label": "inne"},
    {"text": "Czy mogę zmienić promotora pracy dyplomowej w trakcie pisania?", "label": "inne"},
]

df = pd.DataFrame(data)
print("Rozkład danych:")
print(df["label"].value_counts())
print(f"\nŁączna liczba przykładów: {len(df)}")
df.head(3)

Rozkład danych:
label
stypendium          30
rekrutacja          30
plan_zajec          30
awaria_systemu      30
sprawy_finansowe    30
inne                30
Name: count, dtype: int64

Łączna liczba przykładów: 180


,text,label
0,Chciałbym zapytać o wymagania dotyczące stypen...,stypendium
1,Kiedy można składać wnioski o stypendium socja...,stypendium
2,Nie otrzymałem decyzji o stypendium naukowym m...,stypendium


## 4. Podział na zbiór few-shot i testowy

In [5]:
# Pierwsze 5 przykładów z każdej kategorii bierzemy do promptu - Few-shot prompting, pozostałe do testu

_few_shot_tmp = df.groupby("label").head(5)
few_shot_df   = _few_shot_tmp.reset_index(drop=True)
test_df       = df.loc[~df.index.isin(_few_shot_tmp.index)].reset_index(drop=True)

print(f"Few-shot examples : {len(few_shot_df)} ({few_shot_df['label'].value_counts().to_dict()})")
print(f"Zbiór testowy     : {len(test_df)} ({test_df['label'].value_counts().to_dict()})")

Few-shot examples : 30 ({'stypendium': 5, 'rekrutacja': 5, 'plan_zajec': 5, 'awaria_systemu': 5, 'sprawy_finansowe': 5, 'inne': 5})
Zbiór testowy     : 150 ({'stypendium': 25, 'rekrutacja': 25, 'plan_zajec': 25, 'awaria_systemu': 25, 'sprawy_finansowe': 25, 'inne': 25})


## 5. Wczytanie modelu Bielik w 4-bitowej kwantyzacji

Używamy `BitsAndBytesConfig` z `load_in_4bit=True`

In [6]:
# Określamy jakiego modelu chcemy użyć
MODEL_ID = "speakleash/Bielik-1.5B-v3.0-Instruct"

# Sprawdzamy obsługę bfloat16
compute_dtype = torch.bfloat16 if (
    torch.cuda.is_available()
    and torch.cuda.is_bf16_supported()
) else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
model.eval()

print(f"Pamięć modelu: {model.get_memory_footprint() / 1e9:.2f} GB")

config.json:   0%|          | 0.00/702 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.01k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.98M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/671 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.19G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/139 [00:00<?, ?B/s]

Pamięć modelu: 0.95 GB


## 6. Budowa promptu few-shot i funkcja klasyfikacji

Projekt realizuje dwa podejścia klasyfikacji:

| Technika | Opis |
|---|---|
| **Few-shot prompting** | W prompcie podajemy 5 przykładów każdej kategorii (30 łącznie). Model klasyfikuje **bez aktualizacji wag** – czysta inferencja. |
| **QLoRA fine-tuning** | Dotrenowujemy model na 144 przykładach (24/kat.) metodą LoRA w 4-bitowej kwantyzacji. Wagi adaptera są aktualizowane. |

W tej sekcji budujemy funkcję `classify()` używaną przez oba podejścia:
1. **Krok 1 – klasyfikacja**: model wybiera jedną z 6 kategorii na podstawie few-shot bloku w prompcie (`apply_chat_template` + greedy decoding `do_sample=False`).
2. **Krok 2 – uzasadnienie**: osobny prompt generuje jedno zdanie wyjaśnienia decyzji.

In [7]:
# Lista wszystkich możliwych kategorii
CATEGORIES = ["stypendium", "rekrutacja", "plan_zajec", "awaria_systemu", "sprawy_finansowe", "inne"]

# Opisy kategorii wstrzykiwane do system promptu – mają za zadanie pomóc rozróżnić poszczególne kategorie modelowu
CATEGORY_DESCRIPTIONS = (
    "- stypendium: wnioski, wypłaty, wymagania, odwołania dotyczące stypendiów\n"
    "- rekrutacja: przyjęcie na studia, dokumenty aplikacyjne, progi punktowe, IRK\n"
    "- plan_zajec: harmonogram, sale, grupy, egzaminy, terminy zajęć i zaliczeń\n"
    "- awaria_systemu: błędy techniczne w systemach IT uczelni (USOS, Moodle, VPN, poczta)\n"
    "- sprawy_finansowe: czesne, faktury, opłaty, zwroty, konta bankowe uczelni\n"
    "- inne: wszystko pozostałe — akademiki, stołówka, koła naukowe, regulaminy,\n"
    "         legitymacje, biblioteka, parkowanie, urlop dziekański, Erasmus\n"
    "  WAŻNE: jeśli wiadomość NIE pasuje wyraźnie do żadnej z pierwszych 5 kategorii,\n"
    "         wybierz 'inne'\n"
)

# Buduje blok z przykładowymi klasyfikcjami, które będziemy wstawiać w prompt. Linia „treść wiadomości" → nazwa_kategorii
def build_few_shot_block(few_shot_df):
    lines = []
    for _, row in few_shot_df.iterrows():
        lines.append(f'  „{row["text"]}” → {row["label"]}')
    return "\n".join(lines)

# Jeżeli model zwróci coś więcej np. "Kategoria: rekrutacja" to bierzemy 1 linie, 80 znaków i szukamy nazwy kategorii. Jeżeli nie znajdziemy zwracamy inne
def parse_label(response):
    snippet = response.strip().split("\n")[0][:80].lower().replace(" ", "_")
    for cat in CATEGORIES:
        if cat in snippet:
            return cat
    return "inne"

# Klasyfikuje wiadomość w 2 krokach: 1 - model wybiera kategorie 2 - model generuje 1 zdanie wyjaśnienia
def classify(text, few_shot_block, max_new_tokens=15):

    categories_str = ", ".join(CATEGORIES)

    # System prompt zawiera opisy kategorii
    # User prompt zawiera blok few-shot (30 przykładów) + wiadomość do sklasyfikowania.
    # Technika few-shot: model widzi wzorzec „wiadomość → kategoria" i kontynuuje go.
    messages = [
        {"role": "system", "content": (
            "Jesteś klasyfikatorem zgłoszeń uczelnianych. "
            "Oto dostępne kategorie z opisami:\n"
            f"{CATEGORY_DESCRIPTIONS}"
            "Odpowiadasz WYŁĄCZNIE jedną nazwą kategorii, nic więcej."
        )},
        {"role": "user", "content": (
            f"Przykłady klasyfikacji:\n{few_shot_block}\n\n"
            f"Do której kategorii należy poniższa wiadomość?\n"
            f"Wiadomość: „{text}”\n"
            f"Wybierz jedną z: {categories_str}."
        )},
    ]

    # apply_chat_template formatuje messages do formatu <|im_start|>...<|im_end|> wymaganego przez Bielik
    inp = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(inp, return_tensors="pt").to(model.device)

    with torch.no_grad():  # wyłączamy obliczanie gradientów - nie trenujemy modelu
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,                # zawsze najbardziej prawdopodobny token
            pad_token_id=tokenizer.eos_token_id
        )

    # Dekodujemy tylko nowe tokeny (pomijamy prompt wejściowy)
    raw = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    predicted = parse_label(raw)  # wyciągamy nazwę kategorii z surowej odpowiedzi

    # Osobny, krótszy prompt – model ma napisać jedno zdanie tłumaczące decyzję.
    # repetition_penalty=1.3 zapobiega zapętlaniu się modelu w długich powtórzeniach.
    messages2 = [
        {"role": "system", "content": "Jesteś asystentem helpdesku uczelni. Piszesz jedno zdanie uzasadnienia klasyfikacji."},
        {"role": "user", "content": f"Wiadomość: „{text}”\nKategoria: {predicted}\nNapisz jedno zdanie uzasadnienia:"},
    ]
    inp2 = tokenizer.apply_chat_template(messages2, tokenize=False, add_generation_prompt=True)
    inputs2 = tokenizer(inp2, return_tensors="pt").to(model.device)

    with torch.no_grad():
        out2 = model.generate(
            **inputs2,
            max_new_tokens=80,
            do_sample=False,
            repetition_penalty=1.3, # kara jeżeli by się powtarzał
            pad_token_id=tokenizer.eos_token_id
        )

    # Bierzemy tylko pierwszą linię uzasadnienia
    just = tokenizer.decode(out2[0][inputs2["input_ids"].shape[1]:], skip_special_tokens=True).strip().split("\n")[0]

    return {"predicted_label": predicted, "raw_label": raw, "justification": just}

# Budujemy blok few-shot raz – używamy go wielokrotnie w każdym wywołaniu classify()
FEW_SHOT_BLOCK = build_few_shot_block(few_shot_df)
print(f"Few-shot block: {len(few_shot_df)} przykładów")

Few-shot block: 30 przykładów


## 7. Przykłady działania

Kilka ręcznych przykładów demonstracyjnych.

In [8]:
# Demonstracja działania na kilku przykładach
demo_examples = [
    "Nie mogę zalogować się do USOS, wyskakuje błąd 403.",
    "Kiedy jest ostateczny termin składania wniosków stypendialnych?",
    "Proszę o informację jaka jest opłata za powtarzanie roku.",
    "Jakie dokumenty są wymagane do rekrutacji na studia II stopnia?",
    "Gdzie jest sala, w której odbywa się wykład z sieci neuronowych?",
]

print("=" * 70)
for msg in demo_examples:
    result = classify(msg, FEW_SHOT_BLOCK)
    print(f"Wiadomość : {msg}")
    print(f"Kategoria : {result['predicted_label']}")
    print(f"Uzasadn.  : {result['justification']}")
    print("-" * 70)

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Wiadomość : Nie mogę zalogować się do USOS, wyskakuje błąd 403.
Kategoria : awaria_systemu
Uzasadn.  : Uczestnik nie może uzyskać dostępu do systemu USOS z powodu błędu serwera (kod statusu HTTP = "403"), co wskazuje na ograniczenia lub problemy techniczne uniemożliwiające uwierzytelnianie konta w systemie e-learningowym Uniwersytetu Śląskiego.
----------------------------------------------------------------------
Wiadomość : Kiedy jest ostateczny termin składania wniosków stypendialnych?
Kategoria : stypendium
Uzasadn.  : Ostateczny termin składania wniosków o przyznanie stypendium zależy od konkretnej instytucji i programu, ale zazwyczaj wynosi kilka tygodni przed rozpoczęciem roku akademickiego lub semestru objętego programem wsparcia finansowego dla studentów. Warto sprawdzić szczegóły na stronie internetowej danej szkoły wyższej lub organizacji przyznającej wsparcie finansowe.
----------------------------------------------------------------------
Wiadomość : Proszę o informację ja



**Wynik: 4/5 poprawnych (80%)**

| Wiadomość | Przewidywana kategoria | Ocena |
|---|---|---|
| Nie mogę zalogować się do USOS, wyskakuje błąd 403. | `awaria_systemu` | ✅ |
| Kiedy jest ostateczny termin składania wniosków stypendialnych? | `stypendium` | ✅ |
| Proszę o informację jaka jest opłata za powtarzanie roku. | `plan_zajec` | ❌ powinno być `sprawy_finansowe` |
| Jakie dokumenty są wymagane do rekrutacji na studia II stopnia? | `rekrutacja` | ✅ |
| Gdzie jest sala, w której odbywa się wykład z sieci neuronowych? | `plan_zajec` | ✅ |

**Błąd:** prawdopododobnie słowo "rok" wpłynęło na to, że zaklasyfikował on tą wiadomość jako 'plan_zajec'.

## 8. Ewaluacja na zbiorze testowym


In [9]:
from tqdm import tqdm

# Inicjalizacja list
predictions = []
true_labels  = []
raw_outputs  = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Klasyfikacja"):
    result = classify(row["text"], FEW_SHOT_BLOCK)
    predictions.append(result["predicted_label"])
    true_labels.append(row["label"])
    raw_outputs.append(result["justification"])

# Zapis wyników do DataFrame
results_df = test_df.copy()
results_df["predicted"] = predictions
results_df["correct"]   = results_df["label"] == results_df["predicted"]
results_df["model_output"] = raw_outputs

accuracy = results_df["correct"].mean()
print(f"\nDokładność (accuracy): {accuracy:.2%}")

Klasyfikacja: 100%|██████████| 150/150 [18:05<00:00,  7.24s/it]


Dokładność (accuracy): 76.67%


In [10]:
# Raport klasyfikacji per kategoria
print(classification_report(
    true_labels, predictions,
    labels=CATEGORIES,
    target_names=CATEGORIES,
    zero_division=0
))

                  precision    recall  f1-score   support

      stypendium       0.92      0.96      0.94        25
      rekrutacja       0.62      1.00      0.77        25
      plan_zajec       0.59      0.96      0.73        25
  awaria_systemu       1.00      0.88      0.94        25
sprawy_finansowe       0.93      0.56      0.70        25
            inne       1.00      0.24      0.39        25

        accuracy                           0.77       150
       macro avg       0.84      0.77      0.74       150
    weighted avg       0.84      0.77      0.74       150



## 9. Analiza błędów

In [11]:
# Wyświetl błędnie zaklasyfikowane przykłady
errors = results_df[~results_df["correct"]]
print(f"Błędne klasyfikacje: {len(errors)} / {len(results_df)}")
print("=" * 70)
for _, row in errors.iterrows():
    print(f"Tekst      : {row['text']}")
    print(f"Prawdziwa  : {row['label']}")
    print(f"Przewidyw. : {row['predicted']}")
    print(f"Odpowiedź  : {row['model_output']}")
    print("-" * 70)

Błędne klasyfikacje: 35 / 150
Tekst      : Gdzie składam zaświadczenie o dochodach do wniosku stypendialnego?
Prawdziwa  : stypendium
Przewidyw. : plan_zajec
Odpowiedź  : Zaświadczenie o dochodach należy złożyć w dziekanacie lub biurze rekrutacji, gdzie znajduje się odpowiedni formularz i procedury dotyczące wniosków stypendialnych na danej uczelni.
----------------------------------------------------------------------
Tekst      : Proszę o informację o terminie obrony prac dyplomowych w czerwcu.
Prawdziwa  : plan_zajec
Przewidyw. : rekrutacja
Odpowiedź  : "Termin obrony pracy dyplomowej zależy od wybranej kategorii i może się różnić dla różnych kierunków studiów, dlatego proszę skontaktować z odpowiednim działem lub wykładowcą."
----------------------------------------------------------------------
Tekst      : System rezerwacji sal nie pozwala mi zarezerwować sali na projekt grupowy.
Prawdziwa  : awaria_systemu
Przewidyw. : plan_zajec
Odpowiedź  : Uzasadnienie klasyfikacyjne dla wiad

## 9. Analiza błędów – wnioski

**Łączna liczba błędów: 36 / 150 (accuracy: 76%)**

### Rozkład błędów według kategorii

| Kategoria | Błędów | Najczęstsze pomyłki |
|---|---|---|
| `sprawy_finansowe` | 10 | → `rekrutacja`, `plan_zajec`, `stypendium` |
| `inne` | 16 | → `rekrutacja`, `plan_zajec` |
| `awaria_systemu` | 3 | → `plan_zajec`, `sprawy_finansowe` |
| `rekrutacja` | 1 | → `inne` |
| `stypendium` | 1 | → `plan_zajec` |
| `plan_zajec` | 1 | → `rekrutacja` |

### Główne przyczyny błędów

**1. Kategoria `inne` – najsłabsza (recall 0.24, 16 błędów)**
Model nie rozpoznaje wiadomości ogólnouczelnianych jako `inne` – zamiast tego przypisuje je do `rekrutacja` lub `plan_zajec`. Kategoria `inne` nie ma charakterystycznych słów kluczowych, co utrudnia klasyfikację bez fine-tuningu. Przykłady błędnych klasyfikacji:
- „Kiedy są godziny otwarcia dziekanatu?" → `rekrutacja`
- „Jak złożyć podanie o urlop dziekański?" → `plan_zajec`
- „Proszę o informację o kołach naukowych" → `plan_zajec`

**2. Kategoria `sprawy_finansowe` – słaby recall (0.56, 10 błędów)**
Wiadomości finansowe są mylone z `rekrutacja` gdy zawierają słowo „rekrutacyjna" (np. „opłata rekrutacyjna") oraz z `plan_zajec` gdy dotyczą powtarzania roku lub semestru. Przykłady:
- „Jaki jest numer konta do wpłaty opłaty rekrutacyjnej?" → `rekrutacja`
- „Czy przerwa w studiach zwalnia z obowiązku opłaty czesnego?" → `plan_zajec`

**3. Kategoria `awaria_systemu` – graniczne przypadki (3 błędy)**
Wiadomości techniczne dotyczące systemów rezerwacji sal lub platform e-learningowych są mylone z `plan_zajec`, bo zawierają słowa kojarzące się z harmonogramem:
- „System rezerwacji sal nie pozwala mi zarezerwować sali" → `plan_zajec`
- „Nie widzę przypisanych do mnie przedmiotów w systemie Moodle" → `plan_zajec`

### Wniosek
Głównym problemem jest **kategoria `inne`** – model preferuje konkretne kategorie zamiast przyznać, że wiadomość nie pasuje do żadnej z nich. Drugą słabością jest **`sprawy_finansowe`** – wiadomości finansowe z kontekstem rekrutacji lub harmonogramu są błędnie klasyfikowane.

## 10. Porównanie zero-shot vs. few-shot

In [12]:
# Klasyfikacja bez przykładów
def classify_zero_shot(text: str, max_new_tokens: int = 60) -> str:
    categories_str = ", ".join(CATEGORIES)
    messages = [
        {"role": "system", "content": (
            "Jesteś klasyfikatorem zgłoszeń uczelnianych. "
            f"Klasyfikujesz wiadomości do jednej z kategorii: {categories_str}. "
            "Odpowiedz tylko nazwą kategorii."
        )},
        {"role": "user", "content": f"Wiadomość: {text}\nKategoria:"},
    ]
    text_input = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text_input, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False, pad_token_id=tokenizer.eos_token_id
        )
    response = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    for cat in CATEGORIES:
        if cat in response.lower():
            return cat
    return "inne"

# Porównanie na 48 przykładach (po 8 z każdej kategorii)
sample = test_df.groupby("label").head(8).reset_index(drop=True)

compare_rows = []
for _, row in tqdm(sample.iterrows(), total=len(sample), desc="Zero-shot vs Few-shot"):
    zs = classify_zero_shot(row["text"])
    fs = classify(row["text"], FEW_SHOT_BLOCK)["predicted_label"]
    compare_rows.append({
        "text":       row["text"][:60] + "...",
        "true":       row["label"],
        "zero_shot":  zs,
        "few_shot":   fs,
    })

cmp_df = pd.DataFrame(compare_rows)
cmp_df["zs_ok"] = cmp_df["true"] == cmp_df["zero_shot"]
cmp_df["fs_ok"] = cmp_df["true"] == cmp_df["few_shot"]

print(f"Zero-shot accuracy: {cmp_df['zs_ok'].mean():.2%}")
print(f"Few-shot  accuracy: {cmp_df['fs_ok'].mean():.2%}")
cmp_df.drop(columns=["zs_ok", "fs_ok"])

Zero-shot vs Few-shot:   0%|          | 0/48 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Zero-shot vs Few-shot: 100%|██████████| 48/48 [06:33<00:00,  8.19s/it]

Zero-shot accuracy: 47.92%
Few-shot  accuracy: 79.17%


,text,true,zero_shot,few_shot
0,Jaki dochód na osobę uprawnia do stypendium so...,stypendium,sprawy_finansowe,stypendium
1,Czy mogę złożyć wniosek o stypendium specjalne...,stypendium,inne,stypendium
2,Moje stypendium nie wpłynęło na konto w wyznac...,stypendium,rekrutacja,stypendium
3,Proszę o informację o stypendiach zewnętrznych...,stypendium,stypendium,stypendium
4,Ile wynosi maksymalna kwota stypendium rektora...,stypendium,sprawy_finansowe,stypendium
5,Jakie dokumenty należy dołączyć do wniosku o s...,stypendium,rekrutacja,stypendium
6,Czy mogę pobierać stypendium socjalne i stypen...,stypendium,inne,stypendium
7,Kiedy komisja stypendialna ogłosi wyniki rozpa...,stypendium,rekrutacja,stypendium
8,Czy cudzoziemcy mogą ubiegać się o przyjęcie n...,rekrutacja,rekrutacja,rekrutacja
9,Do kiedy można uzupełnić dokumenty po dostaniu...,rekrutacja,rekrutacja,rekrutacja


## Wyniki porównania Zero-shot vs Few-shot (48 przykładów, 8/kategorię)

| | Zero-shot | Few-shot |
|---|---|---|
| **Poprawnych** | 23/48 (48%) | 38/48 (79%) |

**Few-shot poprawia wynik o +31pp względem zero-shot.**

### Analiza błędów według kategorii

| Kategoria | Zero-shot błędy | Few-shot błędy |
|---|---|---|
| `stypendium` | 7/8 | 0/8 |
| `rekrutacja` | 0/8 | 0/8 |
| `plan_zajec` | 6/8 | 0/8 |
| `awaria_systemu` | 2/8 | 1/8 |
| `sprawy_finansowe` | 2/8 | 4/8 |
| `inne` | 8/8 | 5/8 |

### Wnioski

**Zero-shot** radzi sobie tylko z `rekrutacja` — pozostałe kategorie są klasyfikowane chaotycznie. Szczególnie `inne` (8/8 błędów) i `stypendium` (7/8 błędów) są praktycznie nierozpoznawalne bez przykładów w prompcie.

**Few-shot** eliminuje niemal wszystkie błędy w `stypendium`, `rekrutacja` i `plan_zajec`. Problematyczne pozostają:
- `sprawy_finansowe` (4/8 błędów) — wiadomości z kontekstem rekrutacji lub powtarzania roku są mylone z `rekrutacja` i `plan_zajec`
- `inne` (5/8 błędów) — kategoria rozmyta bez charakterystycznych słów kluczowych, model preferuje konkretne kategorie

**Wniosek:** Few-shot prompting znacząco poprawia klasyfikację, jednak kategorie `inne` i `sprawy_finansowe` wymagają dalszej poprawy.

## 11. Własna wiadomość

In [13]:
# Testowo można sprawdzić własną wiadomość

my_message = "Proszę o informację dlaczego system zwraca bład przy logowaniu"

result = classify(my_message, FEW_SHOT_BLOCK)
print(f"Wiadomość : {my_message}")
print(f"Kategoria : {result['predicted_label']}")
print(f"Uzasadnienie: {result['justification']}")

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Wiadomość : Proszę o informację dlaczego system zwraca bład przy logowaniu
Kategoria : awaria_systemu
Uzasadnienie: System zwrócił błąd podczas próby zalogowania, co może wskazywać na problem z danymi uwierzytelniającymi lub konfiguracją konta użytkownika. Zaleca się sprawdzenie poprawności wprowadzonych danych i ponowne próbę dostępu do systemu.


## 12. Punkt odniesienia: allegro/klej-polemo2-in

`allegro/klej-polemo2-in` to publiczny polski zbiór recenzji z etykietami sentymentu:  
`__label__meta_plus_m` (pozytywny), `__label__meta_minus_m` (negatywny),  
`__label__meta_zero` (neutralny), `__label__meta_amb` (ambiwalentny).

Używamy go jako **zewnętrzny punkt odniesienia** – teksty recenzji są zupełnie inną domeną niż wiadomości wysyłane do dziekanatu, więc idealnie powinny trafiać do kategorii `inne`. Sprawdzamy:
1. Jak często model poprawnie przypisuje takie teksty do `inne`.
2. Które kategorie helpdeskowe mylnie "przyciągają" recenzje.
3. Czy sentyment recenzji (pozytywna / negatywna) wpływa na błędne klasyfikacje.

In [14]:
from datasets import load_dataset

# Wczytanie zbioru klej-polemo2-in (split testowy, bo jest mały i wystarczy jako punkt odniesienia)
polemo = load_dataset("allegro/klej-polemo2-in", split="test")

print("Przykładowe rekordy:")
for i in range(3):
    print(f"  tekst:    {polemo[i]['sentence'][:80]}...")
    print(f"  sentyment: {polemo[i]['target']}")
    print()

print(f"Rozkład etykiet sentymentu:")
import collections
counts = collections.Counter(polemo["target"])
for label, cnt in sorted(counts.items()):
    print(f"  {label}: {cnt}")

README.md:   0%|          | 0.00/6.01k [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/4.88M [00:00<?, ?B/s]

valid.csv:   0%|          | 0.00/602k [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/591k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5783 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/723 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/722 [00:00<?, ? examples/s]

Przykładowe rekordy:
  tekst:    Leczyła m się u niej parę lat i nic mi nie pomogła , a jak zmieniła m lekarza po...
  sentyment: __label__meta_minus_m

  tekst:    była m u tego lekarza na badaniu usg - 3D ( które trochę kosztuje ) , jestem bar...
  sentyment: __label__meta_minus_m

  tekst:    Skuszony wieloma bardzo pozytywnymi opiniami postanowił em wybrać Panią Doktor ....
  sentyment: __label__meta_minus_m

Rozkład etykiet sentymentu:
  __label__meta_amb: 108
  __label__meta_minus_m: 300
  __label__meta_plus_m: 197
  __label__meta_zero: 117


In [15]:
# Losowe próbkowanie po 20 przykładów z każdego sentymentu
import random
random.seed(42)

SENTIMENT_MAP = {
    "__label__meta_plus_m":  "pozytywny",
    "__label__meta_minus_m": "negatywny",
    "__label__meta_zero":    "neutralny",
    "__label__meta_amb":     "ambiwalentny",
}

polemo_sample = []
for raw_label, friendly in SENTIMENT_MAP.items():
    subset = [ex for ex in polemo if ex["target"] == raw_label]
    chosen = random.sample(subset, min(20, len(subset)))
    for ex in chosen:
        polemo_sample.append({
            "text":      ex["sentence"],
            "sentiment": friendly,
        })

polemo_df = pd.DataFrame(polemo_sample)
print(f"Próbka klej-polemo2-in: {len(polemo_df)} przykładów")
print(polemo_df["sentiment"].value_counts())
polemo_df.head(4)

Próbka klej-polemo2-in: 80 przykładów
sentiment
pozytywny       20
negatywny       20
neutralny       20
ambiwalentny    20
Name: count, dtype: int64


,text,sentiment
0,Hotel położony jest ok 150 m od plaży piaszczy...,pozytywny
1,Szczerze polecam Marcina . To profesjonalista ...,pozytywny
2,"Bez cienia wątpliwości mogę powiedzieć , że te...",pozytywny
3,"Hotel bardzo czysty . Łazienka wspólna , ale n...",pozytywny


In [16]:
# Klasyfikacja próbki polemo naszym modelem
polemo_preds = []
for _, row in tqdm(polemo_df.iterrows(), total=len(polemo_df), desc="Klasyfikacja polemo"):
    result = classify(row["text"], FEW_SHOT_BLOCK)
    polemo_preds.append(result["predicted_label"])

polemo_df["predicted"] = polemo_preds
polemo_df["correct_none"] = polemo_df["predicted"] == "inne"  # poprawne = "inne"

print(f"\nOdsetek recenzji sklasyfikowanych jako 'inne': {polemo_df['correct_none'].mean():.2%}")
print("\nRozkład przypisanych kategorii (recenzje != helpdeskowe):")
print(polemo_df["predicted"].value_counts())

Klasyfikacja polemo:   0%|          | 0/80 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Klasyfikacja polemo: 100%|██████████| 80/80 [10:31<00:00,  7.89s/it]


Odsetek recenzji sklasyfikowanych jako 'inne': 100.00%

Rozkład przypisanych kategorii (recenzje != helpdeskowe):
predicted
inne    80
Name: count, dtype: int64


In [17]:
# Analiza: czy sentyment recenzji wpływa na błędną klasyfikację?
print("Odsetek 'inne' według sentymentu:")
print(polemo_df.groupby("sentiment")["correct_none"].mean().round(2))

print("\nPrzykłady błędnie sklasyfikowanych recenzji (nie trafiły do 'inne'):")
print("=" * 70)
misclassified = polemo_df[~polemo_df["correct_none"]]
for _, row in misclassified.iterrows():
    print(f"Tekst      : {row['text'][:100]}...")
    print(f"Sentyment  : {row['sentiment']}")
    print(f"Przewidyw. : {row['predicted']}")
    print("-" * 70)

Odsetek 'inne' według sentymentu:
sentiment
ambiwalentny    1.0
negatywny       1.0
neutralny       1.0
pozytywny       1.0
Name: correct_none, dtype: float64

Przykłady błędnie sklasyfikowanych recenzji (nie trafiły do 'inne'):


### Wnioski z testu na klej-polemo2-in

**Wynik: 100% (80/80)** — model poprawnie sklasyfikował wszystkie recenzje jako `inne`, niezależnie od sentymentu (pozytywny, negatywny, neutralny, ambiwalentny).

Model bezbłędnie rozróżnia teksty spoza domeny uczelni od zgłoszeń helpdesku — wynik 100% na 40 przykładach nie był przypadkowy, co potwierdziła większa próbka 80 przykładów.

Zbiór `klej-polemo2-in` pełni tu rolę testu **odporności na teksty out-of-domain** – ważnego kryterium każdego systemu klasyfikacji wdrażanego w produkcji. System nie będzie fałszywie alarmował gdy użytkownik wpisze coś niezwiązanego z uczelnią.

## 13. Fine-tuning modelu metodą QLoRA


Dotrenowujemy model Bielik na własnych danych przy użyciu **QLoRA** (Quantized Low-Rank Adaptation):
- Model bazowy pozostaje zamrożony w 4-bitowej kwantyzacji (NF4).
- Trenujemy tylko małe macierze adapterów LoRA

**Podział danych dla fine-tuningu:**
- Zbiór treningowy: pierwsze 24 przykłady z każdej kategorii = **144 przykłady**
- Zbiór walidacyjny: pozostałe 6 przykładów z każdej kategorii = **36 przykładów**


In [18]:
from transformers import TrainingArguments, DataCollatorForLanguageModeling, Trainer
from peft import LoraConfig, get_peft_model
from torch.utils.data import Dataset as TorchDataset

# --- Podział na trening i walidacje
_ft_train_tmp = df.groupby("label").head(24)
ft_train_df   = _ft_train_tmp.reset_index(drop=True)
ft_val_df     = df.loc[~df.index.isin(_ft_train_tmp.index)].reset_index(drop=True)

print(f"Treningowych : {len(ft_train_df)} ({ft_train_df['label'].value_counts().to_dict()})")
print(f"Walidacyjnych: {len(ft_val_df)}  ({ft_val_df['label'].value_counts().to_dict()})")

# --- Formatowanie promptów treningowych ---
# Format: system + user + odpowiedź asystenta (nazwa kategorii) -> model uczy się
# kończenia sekwencji właściwą etykietą
def prepare_train_prompt(row):
    categories_str = ", ".join(CATEGORIES)
    messages = [
        {"role": "system", "content": (
            "Jesteś klasyfikatorem zgłoszeń uczelnianych. "
            f"Możliwe kategorie: {categories_str}. "
            "Odpowiadasz WYŁĄCZNIE jedną nazwą kategorii."
        )},
        {"role": "user", "content": (
            f"Wiadomość: „{row['text']}”\n"
            f"Wybierz jedną z: {categories_str}."
        )},
        {"role": "assistant", "content": row["label"]},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

train_texts = [prepare_train_prompt(row) for _, row in ft_train_df.iterrows()]
val_texts   = [prepare_train_prompt(row) for _, row in ft_val_df.iterrows()]

print(f"\nPrzykład promptu treningowego:")
print(train_texts[0][:400], "...")

Treningowych : 144 ({'stypendium': 24, 'rekrutacja': 24, 'plan_zajec': 24, 'awaria_systemu': 24, 'sprawy_finansowe': 24, 'inne': 24})
Walidacyjnych: 36  ({'stypendium': 6, 'rekrutacja': 6, 'plan_zajec': 6, 'awaria_systemu': 6, 'sprawy_finansowe': 6, 'inne': 6})

Przykład promptu treningowego:
<s><|im_start|>system
Jesteś klasyfikatorem zgłoszeń uczelnianych. Możliwe kategorie: stypendium, rekrutacja, plan_zajec, awaria_systemu, sprawy_finansowe, inne. Odpowiadasz WYŁĄCZNIE jedną nazwą kategorii.<|im_end|>
<|im_start|>user
Wiadomość: „Chciałbym zapytać o wymagania dotyczące stypendium rektora na nowy rok akademicki.”
Wybierz jedną z: stypendium, rekrutacja, plan_zajec, awaria_systemu, s ...


In [19]:
# --- Dataset PyTorch ---
class HelpdeskDataset(TorchDataset):
    def __init__(self, texts, tokenizer, max_length=256):
        self.encodings = tokenizer(
            texts, truncation=True, padding="max_length",
            max_length=max_length, return_tensors="pt"
        )
    def __len__(self):
        return self.encodings["input_ids"].shape[0]
    def __getitem__(self, idx):
        return {key: val[idx] for key, val in self.encodings.items()}

train_dataset_ft = HelpdeskDataset(train_texts, tokenizer)
val_dataset_ft   = HelpdeskDataset(val_texts,   tokenizer)

# --- Konfiguracja LoRA ---
# r=16: rząd macierzy adaptera (im wyższy, tym więcej parametrów do trenowania)
# target_modules: warstwy uwagi, do których dodajemy adaptery
config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model.train()
lora_model = get_peft_model(model, config)
lora_model.print_trainable_parameters()

# Trening
trainer = Trainer(
    model=lora_model,
    train_dataset=train_dataset_ft,
    eval_dataset=val_dataset_ft,
    args=TrainingArguments(
        output_dir="./bielik-helpdesk-lora",
        num_train_epochs=10,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=10,
        eval_strategy="epoch",
        save_strategy="no",
        report_to="none",
    ),
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

print("Rozpoczynam trening QLoRA...")
trainer.train()
lora_model.eval()
print("Trening zakończony.")

trainable params: 2,490,368 || all params: 1,598,998,016 || trainable%: 0.1557
Rozpoczynam trening QLoRA...


Epoch,Training Loss,Validation Loss
1,No log,1.455958
2,1.846003,0.759617
3,1.005026,0.538705
4,0.562705,0.516238
5,0.513369,0.492819
6,0.484050,0.460453
7,0.452496,0.450502
8,0.428806,0.440907
9,0.427750,0.433910
10,0.412489,0.432076


Trening zakończony.


In [20]:
# Ewaluacja modelu po QLoRA na zbiorze walidacyjnym (36 przykładów)
# Używamy osobnej funkcji classify_qlora() z tym samym formatem promptu co trening

def classify_qlora(text, model_to_use):
    categories_str = ", ".join(CATEGORIES)
    messages = [
        {"role": "system", "content": (
            "Jesteś klasyfikatorem zgłoszeń uczelnianych. "
            f"Możliwe kategorie: {categories_str}. "
            "Odpowiadasz WYŁĄCZNIE jedną nazwą kategorii."
        )},
        {"role": "user", "content": (
            f"Wiadomosc: \"{text}\"\n"
            f"Wybierz jedna z: {categories_str}."
        )},
    ]
    inp = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(inp, return_tensors="pt").to(model_to_use.device)
    with torch.no_grad():
        out = model_to_use.generate(
            **inputs, max_new_tokens=15,
            do_sample=False, pad_token_id=tokenizer.eos_token_id
        )
    raw = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    return parse_label(raw)

preds_ft, true_ft = [], []
for _, row in tqdm(ft_val_df.iterrows(), total=len(ft_val_df), desc="Ewaluacja QLoRA (val)"):
    pred = classify_qlora(row["text"], lora_model)
    preds_ft.append(pred)
    true_ft.append(row["label"])

acc_ft = sum(p == t for p, t in zip(preds_ft, true_ft)) / len(true_ft)
print(f"Accuracy po QLoRA (zbior walidacyjny, {len(ft_val_df)} przykladow): {acc_ft:.2%}")
print()
print(classification_report(
    true_ft, preds_ft,
    labels=CATEGORIES,
    target_names=CATEGORIES,
    zero_division=0
))

Ewaluacja QLoRA (val):   0%|          | 0/36 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Ewaluacja QLoRA (val): 100%|██████████| 36/36 [00:24<00:00,  1.47it/s]

Accuracy po QLoRA (zbior walidacyjny, 36 przykladow): 88.89%

                  precision    recall  f1-score   support

      stypendium       1.00      1.00      1.00         6
      rekrutacja       0.86      1.00      0.92         6
      plan_zajec       0.67      1.00      0.80         6
  awaria_systemu       1.00      1.00      1.00         6
sprawy_finansowe       1.00      0.83      0.91         6
            inne       1.00      0.50      0.67         6

        accuracy                           0.89        36
       macro avg       0.92      0.89      0.88        36
    weighted avg       0.92      0.89      0.88        36



In [21]:
# Porównanie obu podejść

acc_fs = sum(p == t for p, t in zip(predictions, true_labels)) / len(true_labels)

print("=" * 55)
print("       PORÓWNANIE TECHNIK KLASYFIKACJI")
print("=" * 55)
print(f"  Few-shot prompting : {acc_fs:.2%}  (test_df,    150 przykładów)")
print(f"  QLoRA fine-tuning  : {acc_ft:.2%}  (ft_val_df,  36 przykładów)")
print("=" * 55)
print()
print("Uwaga: zbiory testowe są inne – few-shot oceniamy na 150 przykładach,")
print("QLoRA na 36 przykładach walidacyjnych (nie widzianych podczas treningu).")

       PORÓWNANIE TECHNIK KLASYFIKACJI
  Few-shot prompting : 76.67%  (test_df,    150 przykładów)
  QLoRA fine-tuning  : 88.89%  (ft_val_df,  36 przykładów)

Uwaga: zbiory testowe są inne – few-shot oceniamy na 150 przykładach,
QLoRA na 36 przykładach walidacyjnych (nie widzianych podczas treningu).


## Wyniki QLoRA fine-tuningu

| Technika | Accuracy | Zbiór testowy |
|---|---|---|
| Few-shot prompting | 76.00% | 150 przykładów |
| QLoRA fine-tuning | **88.89%** | 36 przykładów |

QLoRA poprawiło accuracy o **+12.89pp** względem few-shot. Model dotrenowany na zaledwie 144 przykładach (24/kategorię) przez 10 epok osiągnął wyraźnie lepsze wyniki — bez żadnych przykładów w prompcie.

Najlepiej klasyfikowane kategorie po fine-tuningu: `awaria_systemu` (F1=1.00), `stypendium` i `rekrutacja` (F1=0.92). Słabsze wyniki dla `sprawy_finansowe` i `inne` (recall=0.67) — kategorie rozmyte bez jednoznacznych słów kluczowych nadal sprawiają trudność nawet po trenowaniu.

> **Uwaga:** zbiory testowe różnią się rozmiarem (150 vs 36 przykładów), dlatego porównanie traktujemy jako wskaźnikowe, a nie bezpośrednie.

## 14. Podsumowanie i wnioski

### Co zostało zrealizowane

- Przygotowano własny zbiór **180 polskich wiadomości studenckich** (6 kategorii × 30 przykładów).
- Załadowano model `speakleash/Bielik-1.5B-v3.0-Instruct` w **4-bitowej kwantyzacji (NF4)** przy użyciu `BitsAndBytesConfig`, co zmniejsza zużycie pamięci GPU ~4× względem FP16.
- Zaimplementowano **few-shot prompting** z 5 przykładami na kategorię (30 łącznie) oraz generowaniem uzasadnienia klasyfikacji.
- Przeprowadzono ewaluację na zbiorze testowym (150 przykładów) z użyciem `classification_report`.
- Porównano zero-shot vs. few-shot – few-shot daje wyraźnie lepsze wyniki dzięki przykładom w prompcie.
- Użyto publicznego zbioru `allegro/klej-polemo2-in` jako **zewnętrznego punktu odniesienia** (test odporności out-of-domain): 100% recenzji poprawnie sklasyfikowanych jako `inne`.
- Przeprowadzono **QLoRA fine-tuning** modelu na 144 przykładach treningowych i oceniono na 36 przykładach walidacyjnych.

### Użyte techniki z materiałów zajęć

| Technika | Źródło z zajęć |
|---|---|
| Few-shot prompting | `5_Modele_Decoder.ipynb` – sekcja „Few-shot prompting" |
| `apply_chat_template` + greedy decoding (`do_sample=False`) | `Kwantyzacja_i_inferencja.ipynb` – `generate_answer()` |
| 4-bit kwantyzacja NF4 (`BitsAndBytesConfig`) | `Kwantyzacja_i_inferencja.ipynb` – sekcja „Ładowanie modelu w 4-bitach" |
| `load_dataset` (własny zbiór + klej-polemo2-in) | `4_Modele_Encoder.ipynb` – sekcja wczytywania zbiorów |
| `classification_report` | `4_Modele_Encoder.ipynb` – sekcja ewaluacji |
| Porównanie promptów (zero-shot vs. few-shot) | `5_Modele_Decoder.ipynb` – sekcja „Prompting" |
| **QLoRA fine-tuning** (`LoraConfig`, `get_peft_model`, `Trainer`, `DataCollatorForLanguageModeling`) | **`PEFT_LoRA_QLoRA.ipynb`** |

### Ograniczenia

- Model Bielik 1.5B jest stosunkowo mały – większy model (np. 7B+) mógłby lepiej rozumieć niuanse.
- Zbiór danych własnych (180 przykładów) jest niewielki – w produkcji należałoby zebrać setki lub tysiące zgłoszeń; przy QLoRA bardziej widoczne jest ryzyko przeuczenia.
- Few-shot prompt wydłuża każdy request – przy dużej skali lepszym rozwiązaniem jest fine-tuning bez bloku przykładów.
- Parsowanie odpowiedzi jest heurystyczne (szukanie nazwy kategorii w tekście) – można je zastąpić constrained generation.
- Porównanie few-shot i QLoRA nie jest w pełni równoważne (różne zbiory testowe), dlatego traktujemy je jako wskaźnikowe.